# Matching books by meaning, not spelling

Two readers describe the same novel in different words. One calls it a tale of a *knight* and a
*dragon*; another, a story of a *warrior* and a *monster*. Unit 3's lexical path would score those
two descriptions as near-strangers -- BM25 matches on *shared surface tokens*, and these share
almost none -- yet any reader can see they mean nearly the same thing. Can a retrieval path match on
**meaning** instead of spelling?

This unit builds the book's first **semantic** content path. Instead of representing a book as a
*sparse* bag of the exact words it uses (Unit 3), we represent it as a *dense* vector in a pretrained
**word-embedding** space, where words that mean similar things sit close together. We will meet the
method honestly: on our synthetic catalog the semantic path clears the random floor comfortably but
lands *below* lexical BM25. Its real value is that it surfaces books the other paths *miss*, so it
**complements** them -- which we will measure, not just assert.

It also opens Part 2's big idea. Unit 5's matrix factorization *learned* an embedding per book from
the interaction log; here we *borrow* one GloVe pretrained on a mountain of text; Unit 8 will *learn*
one with a neural network. All three retrieve the same way: a dot product between embeddings.

## Load the catalog, the keywords, and the GloVe subset

The Unit 1-6 substrate is unchanged: `generated_dir()` locates the seeded slice, the catalog and the
gzip'd interaction log load as before, and `load_keywords` gives each book's keyword bag -- the same
text Unit 3's lexical path indexes. New this unit, `load_glove_subset()` loads a small, committed
slice of the pretrained **GloVe** word vectors (Wikipedia + Gigaword, 100 dimensions), restricted to
the 608 words our catalog's keywords use. It covers 607 of them; one word is out of vocabulary,
recorded in the sidecar and skipped wherever it appears. We fit Unit 3's lexical path now as the
comparison baseline; the collaborative paths are fit later, where we need them.

In [ ]:
import json

import numpy as np
import pandas as pd
from bookrec import (
    LexicalRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    book_embedding,
    generated_dir,
    load_catalog,
    load_glove_subset,
    load_keywords,
    run_validation_scoreboard,
)
from bookrec.lexical import tokenize

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

glove = load_glove_subset()
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

covered = len(glove.tokens) - len(glove.missing)
print(f"catalog {len(catalog_ids)} books; {len(rows)} interaction rows")
print(f"GloVe subset: {len(glove.tokens)} vocab words x {glove.dim} dims")
print(f"covered {covered}/{len(glove.tokens)} words; out-of-vocabulary: {glove.missing}")

## 1. From sparse words to dense meaning

Unit 3 represented a book as a **sparse** vector: one dimension per vocabulary word, almost all
zeros, with a weight on each word the book's keywords actually contain. Two books match only when
they share the *exact* same words. A **word embedding** replaces that with a **dense** vector -- here
100 real numbers per word -- learned so that words appearing in similar contexts across a huge text
corpus end up pointing in similar directions. We measure "similar direction" with **cosine
similarity**: the dot product of two L2-normalized vectors, which is 1.0 for identical direction and
near 0.0 for unrelated words.

Because our committed subset holds only the 607 words our catalog uses, we can ask a sharp question:
for a given word, which *other catalog words* does GloVe place nearest? (This is nearest-neighbour
*within our 607-word vocabulary* -- not the full-GloVe analogies you may have seen, which need the
whole 400,000-word space.)

In [ ]:
# Stack the in-vocabulary word vectors into one matrix, each row L2-normalized, for cosine search.
vocab = [token for token in glove.tokens if token not in set(glove.missing)]
word_matrix = np.array([glove.vector(token) for token in vocab])
word_matrix = word_matrix / np.linalg.norm(word_matrix, axis=1, keepdims=True)


def nearest_words(word, n=5):
    target = glove.vector(word)
    target = target / np.linalg.norm(target)
    similarities = word_matrix @ target          # cosine, since every row is unit length
    order = np.argsort(-similarities)
    return [(vocab[i], float(similarities[i])) for i in order if vocab[i] != word][:n]


print("nearest words within the catalog's 607-word vocabulary:")
for word in ["dragon", "wizard", "murder", "detective"]:
    neighbours = nearest_words(word)
    print(f"{word:>10}: " + ", ".join(f"{token} {score:.2f}" for token, score in neighbours))

GloVe has never seen our catalog, yet within our vocabulary it lands `dragon` beside `sword`,
`demon`, and `monster`; `murder` beside `crime`, `suspect`, and `witness`; `wizard` beside
`sorcerer` and `magic`. These are *meaning* relationships, not spelling ones -- `dragon` and `sword`
share no letters, and the lexical path would treat them as unrelated. That is the signal we want: if
a reader likes books about dragons, a semantic path can surface a book about sorcerers even when the
two keyword bags share no exact word.

## 2. A book embedding, a reader embedding, and brute-force cosine retrieval

A word vector describes one word; a book's keyword bag has many. The simplest way to fold many word
vectors into one **book embedding** is to **average** them and L2-normalize the result -- "mean
pooling". Out-of-vocabulary words are skipped, and a book with no in-vocabulary word gets a safe zero
vector. A **reader embedding** is the same move one level up: average the embeddings of the books the
reader has already read, normalized. Retrieval is then **brute force** -- score every catalog book by
the cosine between its embedding and the reader's, drop the books the reader has seen, keep the top
k.

Build one book's embedding by hand first, to see the mechanism.

In [ ]:
sample_id = catalog_ids[0]
sample_tokens = tokenize(keywords[sample_id])

# Mean-pool by hand: average the in-vocabulary word vectors, then L2-normalize.
vectors = [glove.vector(token) for token in sample_tokens if glove.vector(token) is not None]
mean_vector = np.mean(vectors, axis=0)
by_hand = mean_vector / np.linalg.norm(mean_vector)

print(f"book {sample_id} keywords: {keywords[sample_id]}")
print(f"tokens {len(sample_tokens)}, in-vocabulary {len(vectors)}, embedding dim {by_hand.shape[0]}")
print(f"embedding L2 norm: {float(np.linalg.norm(by_hand)):.4f}")

# bookrec.book_embedding does exactly this: mean pooling, L2-norm, OOV skipped, zero-safe.
helper = book_embedding(sample_tokens, glove)
print(f"matches bookrec.book_embedding: {np.allclose(by_hand, helper, atol=1e-6)}")

That dense vector is the whole content representation -- one per book. Now build a reader's embedding
and retrieve by cosine, still by hand: pick a reader who has read three books, pool those three book
embeddings, and score the catalog against the result.

In [ ]:
# Every catalog book's embedding, stacked into one (n_books, dim) matrix.
book_ids = sorted(item_id for item_id in keywords if item_id in set(catalog_ids))
book_matrix = np.array([book_embedding(tokenize(keywords[i]), glove) for i in book_ids])

# A reader who has read three books: their embedding is the normalized mean of those book vectors.
seen = {book_ids[0], book_ids[1], book_ids[2]}
seen_rows = [row for row, item_id in enumerate(book_ids) if item_id in seen]
reader_vector = book_matrix[seen_rows].mean(axis=0)
reader_vector = reader_vector / np.linalg.norm(reader_vector)

# Brute-force cosine: a dot product against every book (rows are unit-norm), drop seen, keep top 10.
scores = book_matrix @ reader_vector
order = np.argsort(-scores)
by_hand_top10 = [book_ids[row] for row in order if book_ids[row] not in seen][:10]

# Reveal: SemanticEmbeddingRetrievalPath packages exactly this -- embeddings at fit, cosine at
# retrieve -- behind the same RetrievalPath contract every other path implements.
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
path_top10 = [candidate.item_id for candidate in semantic.retrieve(0, {"seen": seen}, 10)]

print(f"by-hand   top 10: {by_hand_top10}")
print(f"the path  top 10: {path_top10}")
print(f"identical: {by_hand_top10 == path_top10}")

## 3. Score it honestly on the validation split

Now the real question: how good is semantic retrieval? Score it on the frozen `val` split with the
same scoreboard every path uses -- hit@10, recall, precision, NDCG -- alongside the random floor,
Unit 3's lexical path, and the collaborative paths from Units 4 and 5. (`test` stays sealed for
Checkpoint A; we never score on it here.) We fit the collaborative paths now; the matrix-factorization
fit is the one slow step.

In [ ]:
from bookrec import (
    ItemItemRetrievalPath,
    MatrixFactorizationPath,
    PopularityRetrievalPath,
    RandomRetrievalPath,
)

popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=0).fit(rows, catalog=catalog_ids)
floor = RandomRetrievalPath(catalog_ids, seed=0)

board = {"random": floor, "lexical": lexical, "semantic": semantic,
         "item-item": item_item, "mf": mf}
print(f"{'path':>10} {'hit@10':>8} {'recall':>8} {'prec@10':>8} {'ndcg@10':>8}")
for name, path in board.items():
    result = run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers
    )
    print(f"{name:>10} {result.hit_rate_at_k:>8.4f} {result.recall_at_k:>8.4f} "
          f"{result.precision_at_k:>8.4f} {result.ndcg_at_k:>8.4f}")

Read the semantic row honestly. At hit@10 about **0.102** it clears the random floor (about 0.012)
by roughly **8.5x** -- the method is genuinely finding signal, not guessing. But it sits **below**
lexical BM25 (about 0.158, so semantic is about 65% of it) and far below the collaborative paths,
item-item CF (about 0.252) and MF (about 0.276). On this catalog, matching on meaning is a *weaker*
content signal than matching on exact shared keywords.

Why? It is an artifact of how our synthetic catalog was built, and worth understanding. Our books'
taste structure comes from latent topics that are **arbitrary contiguous slices of a general word
pool** -- so the words inside one latent topic have no particular GloVe-meaning relationship to one
another. Look back at book 0's keywords: `plaque mind feather candelabra playground sparrow ...`, a
grab-bag, not a theme. GloVe's meaning-geometry is therefore **uncorrelated by construction** with
the latent taste the readers actually follow; only the 12 genre word-banks (the `dragon`/`sword`
cluster we saw in Section 1) carry meaning GloVe can exploit. On a *real* catalog -- where a book's
description is coherent prose about its actual subject -- dense semantics would fare very
differently. We teach the *method* honestly on the data we have, not a staged win.

### So why ship it? Complementarity.

A path does not have to be the most accurate to earn its place -- it has to find books the others
*miss*. Measure the overlap between lexical's top 10 and semantic's top 10 for each reader: if they
return the same books, semantic is a redundant, weaker copy of lexical; if they differ, semantic
reaches a different part of the catalog.

In [ ]:
from collections import defaultdict

seen_by_reader = defaultdict(set)
val_by_reader = defaultdict(set)
known = set(catalog_ids)
for row in rows:
    reader_id, item_id = int(row["reader_id"]), int(row["item_id"])
    if item_id not in known or int(row["label"]) != 1:
        continue
    if row["split"] == "train":
        seen_by_reader[reader_id].add(item_id)
    elif row["split"] == "val":
        val_by_reader[reader_id].add(item_id)

overlaps = []
for reader_id in sorted(val_by_reader):
    history = seen_by_reader[reader_id]
    if reader_id in cold_readers or not (val_by_reader[reader_id] - history):
        continue
    semantic_top = {c.item_id for c in semantic.retrieve(reader_id, {"seen": history}, 10)}
    lexical_top = {c.item_id for c in lexical.retrieve(reader_id, {"seen": history}, 10)}
    if semantic_top and lexical_top:
        overlaps.append(len(semantic_top & lexical_top) / 10)

print(f"mean lexical/semantic top-10 overlap: {np.mean(overlaps):.4f} over {len(overlaps)} readers")

About **0.22** -- on average only a couple of each reader's ten semantic recommendations also appear
among their ten lexical ones. The two content paths largely surface **different** books. That is what
a complementary signal looks like: semantic is not a weaker copy of lexical, it reaches elsewhere.
The real test of whether that complementarity *helps* is to drop semantic into the Unit 6 blend and
measure.

### Adding semantic to the Unit 6 blend

Unit 6 built a calibrated, weighted blend of popularity, lexical, item-item, and MF -- each path's
pool rescaled to a common [0, 1] range, then a weighted union, then a re-rank. Add the semantic path
as a fifth component at the same small content weight lexical carries, keep everything else at Unit
6's pinned configuration, and measure hit@10 and catalog coverage with and without it.

In [ ]:
from bookrec import run_blended_scoreboard

u6_paths = {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf}
u6_weights = {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0}

without_semantic = run_blended_scoreboard(
    u6_paths, interactions_path, catalog_ids=catalog_ids,
    k=10, pool=30, weights=u6_weights, cold_readers=cold_readers,
)
with_semantic = run_blended_scoreboard(
    {**u6_paths, "semantic": semantic}, interactions_path, catalog_ids=catalog_ids,
    k=10, pool=30, weights={**u6_weights, "semantic": 0.5}, cold_readers=cold_readers,
)
print("                   hit@10   coverage")
print(f"Unit 6 blend       {without_semantic.hit_rate_at_k:.4f}   {without_semantic.catalog_coverage:.4f}")
print(f"+ semantic         {with_semantic.hit_rate_at_k:.4f}   {with_semantic.catalog_coverage:.4f}")

Adding the semantic path lifts **both** numbers at once: hit@10 edges up and catalog coverage climbs.
That is the payoff of a complementary signal measured directly -- semantic surfaces meaning-related
books the other four paths would never pool, so the blend both ranks a little better *and* reaches
more of the shelf. A path that is only 65% as accurate as lexical on its own still earns its place,
because what it adds is *different*, not *more of the same*. (As always this is a result on *this*
data at *this* pinned config, not a law; we read it once on `val` and leave `test` sealed.)

## Where this goes

- A **word embedding** (GloVe) places words with similar meaning in similar directions; **cosine**
  measures that closeness. Within our 607-word vocabulary GloVe put `dragon` beside `sword` and
  `murder` beside `crime` -- meaning relationships a lexical path cannot see.
- A **book/reader embedding** is mean-pooled, L2-normalized word vectors; **embedding retrieval** is
  brute-force cosine over the catalog. `SemanticEmbeddingRetrievalPath` packages exactly the by-hand
  computation we built, behind the usual `RetrievalPath` contract.
- Measured honestly, semantic (hit@10 about 0.102) clears the floor about 8.5x but lands **below**
  lexical (0.158) and far below CF/MF -- a synthetic-corpus artifact, since our latent topics are
  arbitrary word-pool slices GloVe's meaning cannot track. Its value is **complementarity**: only
  about a 0.22 top-10 overlap with lexical, and adding it to the Unit 6 blend lifts both hit@10 and
  catalog coverage.

### The bridge ahead

Every path in this unit scored by one operation: a **dot product between embeddings**. That is the
thread through Part 2. Unit 5's matrix factorization already did it with embeddings it *learned* from
the interaction log; this unit did it with embeddings GloVe *pretrained* on text; Unit 8 will *learn*
them with a neural two-tower, and Unit 10 will make the brute-force search we did by hand
*approximate* so it scales to millions of books. Same retriever -- the embeddings just keep getting
better.